# 15 - Evaluate a Trained Model

Load the FrozenLake DQN checkpoint and tokenizer from
[02 — Offline DQN](02_train_offline_dqn.ipynb), then evaluate on new maps with cached
inference. The model weights stay fixed throughout evaluation.

The final cells optionally render replay videos. The loading and cached inference
pattern also applies to other environments.

For checkpoints from [25 — Reasoning](25_train_reasoning_dqn.ipynb), run the
standalone sequential reasoning section at the end instead of the standard DQN cells.

Example 23 checkpoints have a separate autoregressive-reasoning section at the end.

## Setup

Imports and configuration. Edit these settings before running the cells below.


In [ ]:
import os

import gymnasium as gym
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display
from mouse_gym import EnvConfig, RepeatTaskTracker, make_group_env
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv
from torch import nn

from mouse_core import load_model, load_tokenizer
from mouse_core.data import EventLog, Tokenizer
from mouse_core.models import (
    LoRAConfig,
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    action_readouts,
    prepare_decode,
    select_discrete_action,
)

# Repositories
# Hugging Face model repo for load_model
MODEL_ID = "mouse-example-model-offline"

# Environment
# max steps per episode
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task
REPEAT_COUNT = 20
MAX_ACTIONS = 4

# Model
# Model — match the implementation and settings used in example 02
PRETRAINED = "Qwen/Qwen3-0.6B"
BACKBONE_LORA = {"rank": 16, "alpha": 32.0, "dropout": 0.0, "dtype": "float32"}
HEAD_LAYERS = 1
HEAD_SCALE = 0.1
# Hugging Face continuous batching
# maximum new tokens processed in one decoder forward
DECODE_MAX_BATCH_TOKENS = 1024
# shared KV capacity across streams, including 16-token page rounding
DECODE_CACHE_CAPACITY_TOKENS = 32768
# Evaluation
# lockstep env steps for inference (passed to run_eval)
EVAL_STEPS = 512
# used only for video score formatting / frame cutoff
EPISODES_PER_ENV = 1
# held-out env seed stream (far from train)
EVAL_SEED_OFFSET = 1000000

# Collection
# number of environment streams in the GroupEnv
NUM_ENVS = 2

## Runtime


In [ ]:
ensure_registered()
# Render videos without requiring a display or audio device.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Build Environment

The model was trained on rows containing `action`, `observation`, `reward`,
`inner_terminated`, and `inner_truncated`. `_terminated` and `_truncated` are objective
columns. The evaluation environment must emit the same fields and record the zero-based
`repeat_episode_index` only on inner reset events, including the first reset. Rendering
is optional and only used here to create replay videos.


In [ ]:
def make_lake(*, stream: int):
    def env_fn():
        lake = gym.make(
            "Procedural-FrozenLake-v1",
            width=8,
            height=8,
            max_episode_steps=MAX_STEPS_PER_EPISODE,
            map_seed=stream,
            slippery_success_rate=1.0,
            permute_obs=True,
            permute_actions=True,
            render_mode="rgb_array",
            emit_q_star=False,
        )
        env = RepeatTaskEnv(
            lake,
            max_task_episodes=REPEAT_COUNT,
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )
        return env

    return env_fn


configs = [
    EnvConfig(
        name=f"proc_frozenlake_{i}",
        seed=i + EVAL_SEED_OFFSET,
        seed_mode="stream",
        env_fn=make_lake(stream=i + EVAL_SEED_OFFSET),
        reset_options={"regenerate_map": True},
    )
    for i in range(NUM_ENVS)
]

eval_env = make_group_env(configs)

logs = [EventLog(name=name) for name in eval_env.names]
trackers = [RepeatTaskTracker() for _ in logs]

## Tokenizer

Load the saved packing specification and verify its value-readout marker.


In [ ]:
tokenizer = load_tokenizer(repo_id_or_path=MODEL_ID, force_download=True)
compute_text = next(
    spec.format
    for spec in tokenizer.input_fields
    if spec.type == "text"
    and spec.provider == "environment"
    and spec.input_field == "action_requested"
)
if not isinstance(compute_text, str) or compute_text != " ." * (len(compute_text) // 2):
    raise ValueError("Expected the fixed compute text saved by example 02")

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.

Reconstruct example 02's encoder, Q head, and architecture settings before `load_model`;
weights do not reconstruct Python modules. Load the tokenizer from the same bundle. PPO
and GRPO checkpoints need their own policy-head architecture and action selection.

Define `action_name` below before loading the tokenizer saved by example 02.
The separator expression and fixed compute text are saved directly in the tokenizer.
Inference reuses that text, so cached and rebuilt histories contain identical tokens.


In [ ]:
class Encoder(Model):
    def __init__(self, *, pretrained, tokenizer_sources, dtype):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            lora=LoRAConfig(**BACKBONE_LORA),
            residual_dtype=torch.float32,
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_mask=tensors["head_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"],
                            max_batch_tokens=DECODE_MAX_BATCH_TOKENS,
                            cache_capacity_tokens=DECODE_CACHE_CAPACITY_TOKENS,
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class QHead(nn.Module):
    def __init__(self, *, rms_norm_eps, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
         rms_norm_eps=rms_norm_eps)

    def forward(self, z):
        return self.mlp(z)


def action_name(action):
    return ("a", "b", "c", "d")[action]


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)
q_head = (
    QHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
     rms_norm_eps=encoder.backbone.rms_norm_eps)
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

if not isinstance(tokenizer, Tokenizer):
    raise ValueError("Inference requires an Tokenizer checkpoint")
load_model(
    model=modules, repo_id_or_path=MODEL_ID, force_download=True, map_location=device
)
modules.eval()

## Run

The encoder uses Hugging Face's continuous-batching scheduler and paged KV allocator
with Mouse's existing embeddings, FlexAttention decoder, and Q head. Long inputs
are split at `DECODE_MAX_BATCH_TOKENS`; decoder layers process only real tokens.
The environment loop here remains lockstep. For independently ready environments,
Core also exposes `session.submit(row, embeds)` and `session.step()`.

`DECODE_CACHE_CAPACITY_TOKENS` bounds the shared pool; it does not change attention.
Training and inference both use full causal attention within each outer episode.
HF retains its pages while an environment waits, then returns them to the shared
pool when that outer episode ends. Inner resets retain history. A new
`episode_index` clears only its own row and restarts rotary positions at zero.
If shared capacity is insufficient, decoding raises rather than evicting history.
Allocation rounds each live episode to HF's default page size. Close the cache
before changing weights.


Decode takes model inputs (`TensorDict`) from `Tokenizer.pack_rows`, using the same
tokenizer and sequence boundaries as training. Completed-episode scores come from
the trackers' completed task summaries.


In [ ]:
eval_contexts = [[] for _ in eval_env.names]


@torch.no_grad()
def run_eval(*, num_steps: int, temperature: float):
    """Greedy inference with frame capture; scores from ``trackers``.

    ``num_steps`` is lockstep ``GroupEnv`` steps for this call.
    Row ``contexts`` persist across the call (mutated in place). The KV cache
    starts from the full current episode, then consumes only new events.
    A new ``episode_index`` clears that stream's context and resets its KV row.
    Completed-episode scores come from ``trackers``.
    """
    modules.eval()
    n = len(eval_env.names)
    if len(eval_contexts) != n:
        raise ValueError(f"contexts has {len(eval_contexts)} streams but env has {n}")
    kv_cache = None
    cached_ends = np.zeros(n, dtype=np.int64)
    inputs = None
    outputs = None
    for tracker in trackers:
        tracker.clear()
    action_space = eval_env.action_space.spaces[0]
    assert isinstance(action_space, gym.spaces.Discrete)
    num_actions = int(action_space.n)
    video_names = eval_env.names
    video_envs = eval_env.envs
    frames_per_env = [[] for _ in video_names]
    steps_done = 0

    def _act_from_contexts() -> list[int | None]:
        nonlocal kv_cache, cached_ends
        ends = np.array([len(c) for c in eval_contexts], dtype=np.int64)
        batch = [eval_contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)]
        with torch.no_grad():
            inputs, inputs_objective_data = tokenizer.pack_rows(
                rows=[[event for step in row for event in step] for row in batch],
            )
            out = encoder(inputs, cache=kv_cache, use_cache=True)
            kv_cache = out["cache"]
            cached_ends = ends.copy()
            requested = [
                i for i, tracker in enumerate(trackers) if tracker.action_requested
            ]
            actions = select_discrete_action(
                scores=action_readouts(
                    q_head(out["z"]), inputs, sequence_ids=requested
                ),
                temperature=temperature,
                num_actions=num_actions,
            )
        chosen = iter(actions.cpu().tolist())
        return [
            next(chosen) if tracker.action_requested else None for tracker in trackers
        ]

    try:
        for _ in range(num_steps):
            event_starts = [len(log) for log in logs]
            for log, tracker in zip(logs, trackers):
                episode_start = tracker.output is None or tracker.ended
                if episode_start:
                    log.event(
                        provider="coordinator",
                        episode_index=tracker.next_episode_index,
                        step_index=(0 if episode_start else tracker.step_index + 1),
                        payload={
                            "_episode_start": True,
                        },
                    )

            if any(tracker.action_requested for tracker in trackers):
                inputs = _act_from_contexts()
            else:
                inputs = [None] * n
            for log, tracker, action in zip(logs, trackers, inputs):
                if action is not None:
                    log.event(
                        provider="agent",
                        episode_index=tracker.next_episode_index,
                        step_index=tracker.step_index + 1,
                        payload={
                            "action": action,
                        },
                    )
            outputs = eval_env.step(tuple(inputs))
            for i, (action, out) in enumerate(zip(inputs, outputs)):
                log = logs[i]
                tracker = trackers[i]
                tracker.update(output=out)

                log.event(
                    provider="environment",
                    episode_index=tracker.episode_index,
                    step_index=tracker.step_index,
                    payload={
                        "observation": tracker.observation,
                        **(
                            {"repeat_episode_index": tracker.repeat_episode_index}
                            if tracker.repeat_episode_start
                            else {}
                        ),
                        **(
                            {"inner_terminated": True}
                            if tracker.inner_terminated
                            else {}
                        ),
                        **(
                            {"inner_truncated": True} if tracker.inner_truncated else {}
                        ),
                        **(
                            {"_q_star": tracker.info["q_star"]}
                            if tracker.action_requested and "q_star" in tracker.info
                            else {}
                        ),
                        **({"_terminated": True} if tracker.terminated else {}),
                        **({"_truncated": True} if tracker.truncated else {}),
                        **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                        **(
                            {"action_requested": True}
                            if tracker.action_requested
                            else {}
                        ),
                    },
                )
                row = log[event_starts[i] :]
                if eval_contexts[i] and tracker.episode_start:
                    eval_contexts[i] = []
                    cached_ends[i] = 0
                    if kv_cache is not None:
                        kv_cache.reset_rows(rows=[i])
                eval_contexts[i].append(row)
            for i, video_env in enumerate(video_envs):
                if eval_contexts[i][-1][-1]["episode_index"] >= EPISODES_PER_ENV:
                    continue
                frame = video_env.render()
                if frame is not None:
                    frames_per_env[i].append(np.asarray(frame).copy())
            steps_done += 1
            if steps_done % 100 == 0:
                episode_scores = [
                    task.reward for tracker in trackers for task in tracker.tasks
                ]
                mean_episode_score = (
                    sum(episode_scores) / len(episode_scores)
                    if episode_scores
                    else float("nan")
                )
                print(
                    f"step {steps_done} | {len(episode_scores)} episodes this window | mean episode score {mean_episode_score:.2f}/{REPEAT_COUNT}",
                )
    finally:
        if kv_cache is not None:
            kv_cache.close()
    scores_per_env = [[task.reward for task in tracker.tasks] for tracker in trackers]
    return (video_names, frames_per_env, scores_per_env)


video_names, frames_per_env, eval_episode_scores = run_eval(
    num_steps=EVAL_STEPS, temperature=0.0
)
eval_env.close()

## Replay MP4s

The remaining cells format environment-specific render frames as MP4s. This is not
required for Mouse Core inference, but it is a useful way to inspect what the loaded
policy is doing.


In [ ]:
def format_episode_scores(*, episode_scores):
    episode_scores = episode_scores[:EPISODES_PER_ENV]
    scores_text = " ".join((f"{score:.0f}" for score in episode_scores))
    summary = f"episode scores [{scores_text}] / {REPEAT_COUNT} | mean {sum(episode_scores) / EPISODES_PER_ENV:.2f}"
    return summary


def display_env_replay(*, name, frames, episode_scores, fps=5):
    print(f"{name}: {format_episode_scores(episode_scores=episode_scores)}")
    if not frames:
        print("  (no frames captured)")
        return
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(w / 100, h / 100))
    ax.axis("off")
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))


for name, frames, episode_scores in zip(
    video_names, frames_per_env, eval_episode_scores
):
    display_env_replay(name=name, frames=frames, episode_scores=episode_scores)

## Evaluate an Example 25 Sequential Reasoning Checkpoint

Run these cells independently of the standard DQN evaluation above. Set
`REASONING_MODEL_ID` to a checkpoint uploaded by example 25. Its saved loss
selects adaptive all-token supervision or fixed-budget last-token supervision;
the saved residual-connection flag independently selects the recurrence.

Inference uses exactly the same reasoner and input sequence as training.
Last-token supervision selects an environment action at the final fixed depth.
All-token supervision permits only `think` before its saved minimum depth, then
selects environment actions or `think`, with `think` masked at the cap. Task KV
caches retain the generated history across inner episode resets and close only
at the outer task boundary. The saved tokenizer consumes the same event fields
as offline DQN; no separate chat formatter is used. Returns contain
raw environment reward. CPU inference is supported; training remains in example 25.


### Setup


In [ ]:
import json
from pathlib import Path

import gymnasium as gym
import numpy as np
import torch
from huggingface_hub import snapshot_download
from torch import nn
from mouse_gym import EnvConfig, RepeatTaskTracker, make_group_env
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv

from mouse_core.data import load_tokenizer
from mouse_core.models import (
    LoRAConfig,
    Backbone,
    Embedding,
    Model,
    MultiEmbedding,
    SequentialReasoner,
    load_model,
)

# Set to your uploaded model repository; None skips this optional section.
REASONING_MODEL_ID = None  # e.g. "your-account/mouse-example-reasoning-dqn"

REASONING_EVAL_EPISODES = 3

REASONING_EVAL_SEED = 1000000

### Runtime


In [ ]:
REASONING_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Tokenizer


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


def encode_reasoning_events(*, events, token_spec):
    """Use the saved training schema directly on new environment events."""
    tokens, _ = token_spec.pack_rows(rows=[events])
    return tokens


### Build Encoder and Heads


In [ ]:
class ReasoningDqn(Model):
    def __init__(self, settings):
        super().__init__()
        if settings["initial_state"] != "zero_feedback_with_learned_input_offset":
            raise ValueError("Reasoning checkpoints require zero-reset feedback and a learned offset after the input MLP")
        if settings["input_projection"] != "gated_mlp_after_rms_norm":
            raise ValueError("Reasoning checkpoints require a gated MLP after input RMSNorm")
        self.reasoning_loss = settings["reasoning_loss"]
        if self.reasoning_loss not in {"all_tokens", "last_token"}:
            raise ValueError("reasoning_loss must be all_tokens or last_token")
        adaptive = self.reasoning_loss == "all_tokens"
        if not adaptive and settings["min_steps"] != settings["max_steps"]:
            raise ValueError("last_token requires equal minimum and maximum reasoning steps")
        self.embedding = MultiEmbedding(
            embeddings={"text": Embedding(pretrained=settings["pretrained"]),
                        "reasoning": Embedding(vocab_size=1, hidden_dim=settings["hidden_dim"])}
        )
        self.backbone = Backbone(lora=LoRAConfig(**settings["backbone_lora"]), residual_dtype=torch.float32, pretrained=settings["pretrained"], dtype=torch.float32)
        self.value_head = nn.Linear(self.backbone.hidden_dim, settings["num_actions"] + int(adaptive))
        self.reasoner = SequentialReasoner(
            hidden_dim=self.backbone.hidden_dim, input_mlp_intermediate_dim=self.backbone.mlp_intermediate_dim,
            input_mlp_output_scale=settings["input_mlp_output_scale"], min_steps=settings["min_steps"], max_steps=settings["max_steps"],
            residual_connections=settings["residual_connections"], wrap_latent=settings["wrap_latent"], adaptive_depth=adaptive,
         rms_norm_eps=self.backbone.rms_norm_eps)

    def forward(self, tokens, *, cache):
        trace = self.reasoner.generate(
            tokens,
            embedding=self.embedding,
            backbone=self.backbone,
            action_values=self.value_head,
            backend="continuous",
            max_active_requests=1,
            cache=cache,
        )
        final = trace.states[torch.arange(len(trace.lengths), device=trace.states.device), trace.lengths]
        values = self.value_head(final)
        return values[..., :-1] if self.reasoner.adaptive_depth else values


def load_sequential_checkpoint(path, *, device):
    source = str(path)
    path = Path(source)
    if not path.is_dir():
        path = Path(snapshot_download(repo_id=source, repo_type="model"))
    settings = json.loads((path / "reasoning_settings.json").read_text())
    policy = ReasoningDqn(settings).to(device).eval()
    load_model(model=policy, repo_id_or_path=path, map_location=device)
    token_spec = load_tokenizer(repo_id_or_path=str(path / "tokenizer"))
    return policy, token_spec, settings

### Run


In [ ]:
def run_reasoning_eval(*, num_episodes: int, seed: int) -> list[float]:
    """Evaluate complete repeated tasks, retaining context across inner resets."""
    ensure_registered()

    def make_lake():
        return RepeatTaskEnv(
            gym.make(**reasoning_settings["environment"], map_seed=seed),
            max_task_episodes=reasoning_settings["repeat_count"],
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )

    env = make_group_env([EnvConfig(
        name="reasoning", seed=seed, seed_mode="stream", env_fn=make_lake,
        reset_options={"regenerate_map": True},
    )])
    returns = []
    tracker = RepeatTaskTracker()
    try:
        for _ in range(num_episodes):
            pending = [{
                "provider": "coordinator", "episode_index": tracker.next_episode_index,
                "step_index": 0, "payload": {"_episode_start": True},
            }]
            cache = reasoning_policy.backbone.decode_session(batch_size=1)
            action, total_reward = None, 0.0
            try:
                while True:
                    if action is not None:
                        pending.append({
                            "provider": "agent", "episode_index": tracker.next_episode_index,
                            "step_index": tracker.step_index + 1, "payload": {"action": action},
                        })
                    tracker.update(output=env.step((action,))[0])
                    total_reward += tracker.reward
                    pending.append({
                        "provider": "environment", "episode_index": tracker.episode_index,
                        "step_index": tracker.step_index,
                        "payload": {
                            "observation": tracker.observation,
                            **({"repeat_episode_index": tracker.repeat_episode_index}
                               if tracker.repeat_episode_start else {}),
                            **({"inner_terminated": True} if tracker.inner_terminated else {}),
                            **({"inner_truncated": True} if tracker.inner_truncated else {}),
                            **({"_terminated": True} if tracker.terminated else {}),
                            **({"_truncated": True} if tracker.truncated else {}),
                            **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                            **({"action_requested": True} if tracker.action_requested else {}),
                        },
                    })
                    if tracker.ended:
                        break
                    if tracker.action_requested:
                        tokens = encode_reasoning_events(events=pending, token_spec=reasoning_tokenizer)
                        values = reasoning_policy.generate(tokens, cache=cache)
                        action = int(values[-1].argmax(-1).item())
                        pending.clear()
                    else:
                        # Keep passive terminal/reset text for the next decision.
                        action = None
            finally:
                cache.close()
            returns.append(total_reward)
    finally:
        env.close()
    return returns


if REASONING_MODEL_ID is not None:
    reasoning_policy, reasoning_tokenizer, reasoning_settings = (
        load_sequential_checkpoint(REASONING_MODEL_ID, device=REASONING_DEVICE)
    )
    reasoning_returns = run_reasoning_eval(
        num_episodes=REASONING_EVAL_EPISODES, seed=REASONING_EVAL_SEED
    )
    print(f"Task returns: {reasoning_returns}; mean reward: {np.mean(reasoning_returns):.3f}")


## Evaluate an Example 23 Autoregressive Checkpoint

This standalone section evaluates the offline vocabulary-reasoning model from
example 23. Set `AUTOREGRESSIVE_MODEL_ID` to its uploaded Hugging Face repository. It uses the
saved prompt, environment, generation limits, vocabulary head, and end-token rule.
It rebuilds the same observed-history prefixes used by that training example.
No evaluation episodes are added to training data.


### Setup


In [ ]:
import json
from pathlib import Path

import gymnasium as gym
import numpy as np
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from mouse_core.data import load_tokenizer
from mouse_core.models import (
    LoRAConfig,
    MLP,
    Backbone,
    Embedding,
    Model,
    MultiEmbedding,
    load_model,
)
from mouse_core.models.autoregressive_reasoner import AutoregressiveReasoner
from mouse_core.models.vocabulary_head import VocabularyHead

AUTOREGRESSIVE_MODEL_ID = None  # e.g. "your-account/mouse-example-autoregressive-dqn"

AUTOREGRESSIVE_EVAL_EPISODES = 3

AUTOREGRESSIVE_EVAL_SEED = 1000000

### Runtime


In [ ]:
AUTOREGRESSIVE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Tokenizer


In [ ]:
def format_autoregressive_observation(*, observation, action, reward, step_index):
    if action is None:
        text = autoregressive_backend.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": autoregressive_settings["prompt_prefix"]
                    + f"State {observation}.\n",
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    else:
        text = f"Action {action}, reward {reward:g}, state {observation}.\n"
    return {
        "provider": "environment",
        "episode_index": 0,
        "step_index": step_index,
        "payload": {"prompt": text + "Value:", "action_requested": True},
    }

### Build Encoder and Heads


In [ ]:
class ReasoningDqn(Model):
    def __init__(self, settings):
        super().__init__()
        self.settings = settings
        self.embedding = MultiEmbedding(
            embeddings={"text": Embedding(pretrained=settings["pretrained"])}
        )
        self.backbone = Backbone(lora=LoRAConfig(**settings["backbone_lora"]), residual_dtype=torch.float32, pretrained=settings["pretrained"], dtype=torch.float32)
        self.value_head = MLP(
            in_features=self.backbone.hidden_dim,
            out_features=settings["num_actions"],
            hidden_dim=self.backbone.hidden_dim,
            num_layers=1,
            scale=0.1,
            propagate_gradient=1.0,
         rms_norm_eps=self.backbone.rms_norm_eps)
        self.reasoner = AutoregressiveReasoner(
            VocabularyHead.from_pretrained(settings["pretrained"])
        )

    def forward(self, tokens, *, cache):
        traces = self.reasoner.generate(
            tokens,
            embedding=self.embedding,
            backbone=self.backbone,
            tokenizer_name="text",
            backend="continuous",
            end_token_id=self.settings["end_token_id"],
            max_new_tokens=self.settings["max_new_tokens"],
            max_active_requests=1,
            cache=cache,
            do_sample=True,
            temperature=self.settings["temperature"],
        )
        return self.value_head(torch.stack([trace.final_hidden for trace in traces]))



def load_autoregressive_checkpoint(path, *, device):
    source = str(path)
    path = Path(source)
    if not path.is_dir():
        path = Path(snapshot_download(repo_id=source, repo_type="model"))
    settings = json.loads((path / "reasoning_settings.json").read_text())
    policy = ReasoningDqn(settings).to(device).eval()
    load_model(model=policy, repo_id_or_path=path, map_location=device)
    backend = AutoTokenizer.from_pretrained(path / "text_tokenizer")
    token_spec = load_tokenizer(
        repo_id_or_path=str(path / "tokenizer"), tokenizers={"text": backend}
    )
    return policy, token_spec, backend, settings

### Run


In [ ]:
def run_autoregressive_eval(*, num_episodes: int, seed: int) -> list[float]:
    """Evaluate the checkpoint loaded by this section."""
    env = gym.make(**autoregressive_settings["environment"])
    returns = []
    try:
        for episode in range(num_episodes):
            observation, _ = env.reset(seed=seed + episode)
            total, step_index = 0.0, 0
            action, reward = None, 0.0
            cache = autoregressive_policy.backbone.decode_session(batch_size=1)
            try:
                while True:
                    event = format_autoregressive_observation(
                        observation=int(observation), action=action,
                        reward=reward, step_index=step_index,
                    )
                    tokens, _ = autoregressive_tokenizer.pack_rows(rows=[[event]])
                    action = int(
                        autoregressive_policy.generate(tokens, cache=cache)[-1].argmax(-1).item()
                    )
                    observation, reward, terminated, truncated, _ = env.step(action)
                    total += float(reward)
                    step_index += 1
                    if terminated or truncated:
                        break
            finally:
                cache.close()
            returns.append(total)
    finally:
        env.close()
    return returns


if AUTOREGRESSIVE_MODEL_ID is not None:
    (
        autoregressive_policy,
        autoregressive_tokenizer,
        autoregressive_backend,
        autoregressive_settings,
    ) = load_autoregressive_checkpoint(
        AUTOREGRESSIVE_MODEL_ID, device=AUTOREGRESSIVE_DEVICE
    )
    autoregressive_returns = run_autoregressive_eval(
        num_episodes=AUTOREGRESSIVE_EVAL_EPISODES, seed=AUTOREGRESSIVE_EVAL_SEED
    )
    print(
        f"Evaluation returns: {autoregressive_returns}; success rate: {np.mean(autoregressive_returns):.1%}"
    )